# Package Creation

This lab turns a small Python module into a distributable package, with `uv`: a `pyproject.toml` that declares its build back-end and a command, an editable install for development, then building the source archive and the wheel, installing them into a fresh environment and shipping a new version. The package, `thermometer`, converts temperatures from degrees Celsius to degrees Fahrenheit with the `to-fahrenheit` command.

The lab runs in Colab as well as in the Codespace. A line that starts with `!` is a shell command, and a cell that starts with `%%writefile <file>` writes its content to that file. The steps build on each other: each one starts from the package of the previous one. The last one, publishing on Test PyPI, is optional and doesn't run with "Run all".

In [ ]:
# @title Launch first: setting up the environment
!rm -rf sample_data
!pip install -q -U uv

## 1. Create the Package Skeleton

With `uv init`, create the skeleton of a library named `thermometer`, with uv's default build back-end, the one of the slides, then move into its folder with the `%cd` magic command. Then display the list of its files (except `.git`) and its `pyproject.toml`.

Where does uv put the package's code? Which table of `pyproject.toml` names the build back-end, and what does it hold?

**Useful documentation**

- https://docs.astral.sh/uv/reference/cli/#uv-init
- https://docs.astral.sh/uv/concepts/projects/init/#libraries
- https://docs.astral.sh/uv/concepts/build-backend/

In [ ]:
# Your code here

### Additional hints

`uv init --lib` creates a library: the code in `src/thermometer/` and a `[build-system]` table, with no command. Without an option, uv creates an installable application, which already declares a command, and `--no-package` a project that is not installed, with no `[build-system]` table. The default back-end is uv's own, `uv_build` (`--build-backend` chooses another one, such as `hatchling`). `find . -type f -not -path "./.git/*"` lists the files without those of the git repository uv creates.

```python
!uv init ... thermometer
%cd thermometer
!find . -type f -not -path "./.git/*"
!cat ...
```

### Solution

In [ ]:
!uv init --lib thermometer

In [ ]:
%cd thermometer

In [ ]:
!find . -type f -not -path "./.git/*"
!cat pyproject.toml

The code is in `src/thermometer/` (the project's name, with `_` instead of `-`): `__init__.py` makes it a package, and `py.typed` tells type checkers that it is annotated. The `[build-system]` table names the back-end (`build-backend = "uv_build"`) and what to install to run it (`requires = ["uv_build>=…,<…"]`, a version range): `uv`, `pip` or any other front-end reads it to build the package, knowing nothing about `uv_build`.

## 2. Write the Code and Declare the Command

Write two modules in the package:

- `src/thermometer/conversion.py`, with a `celsius_to_fahrenheit(celsius: float) -> float` function (F = C × 9 / 5 + 32);
- `src/thermometer/cli.py`, with a `main()` function that converts each of the command line's arguments and prints a line such as `20.0 °C = 68.0 °F`.

Then rewrite `pyproject.toml`, adding a `[project.scripts]` table that declares the `to-fahrenheit` command, calling `main`. Keep the `[project]` and `[build-system]` tables uv put in it (the description, the authors and the rest as you like).

**Useful documentation**

- https://docs.python.org/3/library/sys.html#sys.argv
- https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#creating-executable-scripts
- https://docs.astral.sh/uv/concepts/projects/config/#command-line-interfaces

In [ ]:
# Your code here

### Additional hints

`sys.argv[1:]` gives the command's arguments, strings to convert with `float`. An entry of `[project.scripts]` maps a command's name to `"<module>:<function>"`, here `"thermometer.cli:main"`. Rewriting the whole file with `%%writefile`, rather than appending the table, lets you rerun the cell without duplicating it. Keep the `[build-system]` table.

```python
%%writefile src/thermometer/cli.py
import sys

from thermometer.conversion import celsius_to_fahrenheit


def main() -> None:
  for argument in ...:
    celsius = ...
    print(f"{celsius} °C = {...} °F")
```

```toml
[project.scripts]
to-fahrenheit = "..."
```

### Solution

In [ ]:
%%writefile src/thermometer/conversion.py
def celsius_to_fahrenheit(celsius: float) -> float:
  return celsius * 9 / 5 + 32

In [ ]:
%%writefile src/thermometer/cli.py
import sys

from thermometer.conversion import celsius_to_fahrenheit


def main() -> None:
  for argument in sys.argv[1:]:
    celsius = float(argument)
    print(f"{celsius} °C = {celsius_to_fahrenheit(celsius)} °F")

In [ ]:
%%writefile pyproject.toml
[project]
name = "thermometer"
version = "0.1.0"
description = "Temperature conversions."
readme = "README.md"
requires-python = ">=3.12"
dependencies = []

[project.scripts]
to-fahrenheit = "thermometer.cli:main"

[build-system]
requires = ["uv_build>=0.13.1,<0.14.0"]
build-backend = "uv_build"

## 3. Install the Package in Editable Mode

Install the package in editable mode with `uv pip install`, then run `to-fahrenheit 20 36.6 -40`. What is the conversion of 36.6 °C?

Fix the display in `cli.py`, with one decimal for both temperatures, and run the command again without reinstalling the package. Why does the fix apply?

**Useful documentation**

- https://docs.astral.sh/uv/reference/cli/#uv-pip-install--editable

In [ ]:
# Your code here

### Additional hints

`-e .` installs the current folder's package in editable mode. In Colab and the Codespace, `uv pip install` installs into the system's Python, the notebook's, whose commands folder is in the `PATH`. Computing 36.6 × 9 / 5 + 32 in floats gives `97.88000000000001`; an f-string's `:.1f` specification shows one decimal.

```python
!uv pip install ...
!to-fahrenheit 20 36.6 -40
```

### Solution

In [ ]:
!uv pip install -e .
!to-fahrenheit 20 36.6 -40

In [ ]:
%%writefile src/thermometer/cli.py
import sys

from thermometer.conversion import celsius_to_fahrenheit


def main() -> None:
  for argument in sys.argv[1:]:
    celsius = float(argument)
    print(f"{celsius:.1f} °C = {celsius_to_fahrenheit(celsius):.1f} °F")

In [ ]:
!to-fahrenheit 20 36.6 -40

36.6 °C gives `97.88000000000001 °F`: 36.6 has no exact binary representation. An editable install doesn't copy the code: it leaves a path to `src/` in the environment, so Python reads the sources at each import. The `to-fahrenheit` command, a small script the install created, imports `thermometer.cli` and calls `main`: it sees every change of the code without reinstalling (not those of `pyproject.toml`, such as a new command, which need a reinstall).

## 4. Build the Distributions

With `uv build`, build the package's source archive (*sdist*, `.tar.gz`) and wheel (`.whl`), in `dist/`. List the content of each with the command-line interfaces of the `tarfile` and `zipfile` modules, then extract the wheel into `../wheel` and display its `METADATA` and `entry_points.txt` files.

Which files are in the archive and not in the wheel, and the other way round? Where does the wheel keep the package's version and its command?

**Useful documentation**

- https://docs.astral.sh/uv/reference/cli/#uv-build
- https://docs.python.org/3/library/tarfile.html#command-line-interface
- https://docs.python.org/3/library/zipfile.html#command-line-interface

In [ ]:
# Your code here

### Additional hints

`uv build` builds both by default, the wheel from the archive. `python -m tarfile -l <archive>` and `python -m zipfile -l <wheel>` list the content, `python -m zipfile -e <wheel> <folder>` extracts the wheel (a wheel is a zip archive). The files of `dist/` are named `thermometer-0.1.0.tar.gz` and `thermometer-0.1.0-py3-none-any.whl`, and the wheel's metadata is in its `thermometer-0.1.0.dist-info/` folder.

```python
!uv build
!python -m tarfile -l dist/...
!python -m zipfile -l dist/...
!python -m zipfile -e dist/... ../wheel
!cat ../wheel/.../METADATA ../wheel/.../entry_points.txt
```

### Solution

In [ ]:
!uv build
!python -m tarfile -l dist/thermometer-0.1.0.tar.gz
!python -m zipfile -l dist/thermometer-0.1.0-py3-none-any.whl

In [ ]:
!python -m zipfile -e dist/thermometer-0.1.0-py3-none-any.whl ../wheel
!cat ../wheel/thermometer-0.1.0.dist-info/METADATA
!cat ../wheel/thermometer-0.1.0.dist-info/entry_points.txt

The source archive holds the sources as they are, `pyproject.toml` included, plus a `PKG-INFO`: installing it means building the package, so running its back-end. The wheel is ready to install: its files only have to be copied to their place. It holds the package's code, without `src/` or `pyproject.toml`, and a `.dist-info` folder: `METADATA` repeats the `[project]` table (name, version, required Python version, and the dependencies in `Requires-Dist` lines), `entry_points.txt` the command of `[project.scripts]`, and `RECORD` the list of files. The wheel's name says which Pythons and platforms it is for: `py3-none-any`, pure Python for all of them. PyPI gets both: the wheel to install fast, the archive for the platforms no wheel covers.

## 5. Install the Wheel in a Fresh Environment

With `uv venv`, create a fresh virtual environment in `../wheel-test`, outside the package's folder, install the wheel into it, then run this environment's `to-fahrenheit` command on `100`.

**Useful documentation**

- https://docs.astral.sh/uv/reference/cli/#uv-venv
- https://docs.astral.sh/uv/reference/cli/#uv-pip-install

In [ ]:
# Your code here

### Additional hints

`--clear` empties the environment if it already exists, instead of asking what to do. `uv pip install --python ../wheel-test/bin/python <wheel>` installs into that environment rather than the notebook's, and its commands are in `../wheel-test/bin/`. The command should print `100.0 °C = 212.0 °F`.

```python
!uv venv ... ../wheel-test
!uv pip install --python ... dist/thermometer-0.1.0-py3-none-any.whl
!../wheel-test/bin/to-fahrenheit 100
```

### Solution

In [ ]:
!uv venv --clear ../wheel-test
!uv pip install --python ../wheel-test/bin/python dist/thermometer-0.1.0-py3-none-any.whl
!../wheel-test/bin/to-fahrenheit 100

This is what the user of a published package does, except that `uv` or `pip` download the wheel from PyPI. Here, the environment gets a copy of the wheel's code: it no longer depends on the `src/` folder. `uvx --from dist/thermometer-0.1.0-py3-none-any.whl to-fahrenheit 100` would do the same in a throwaway environment.

## 6. Ship a New Version

Without arguments, `to-fahrenheit` prints nothing and ends with the exit code 0, the one of a success, as the next cell shows.

Make `main`, without arguments, print a usage message on the standard error and return 1, and 0 otherwise. Then compare the command of the editable install and the one of `../wheel-test`. Next, move the package to version 0.2.0 with `uv version`, rebuild it, install the new wheel into `../wheel-test` and check its command.

**Useful documentation**

- https://docs.astral.sh/uv/reference/cli/#uv-version
- https://docs.astral.sh/uv/guides/package/#updating-your-version
- https://docs.python.org/3/library/sys.html#sys.stderr

In [ ]:
!../wheel-test/bin/to-fahrenheit; echo "Exit code: $?"

In [ ]:
# Your code here

### Additional hints

A command's script calls `sys.exit(main())`: the value `main` returns becomes the exit code. `print(..., file=sys.stderr)` writes to the standard error. `uv version --bump minor` goes from 0.1.0 to 0.2.0 in `pyproject.toml`, and `--no-sync` keeps it from creating a `.venv` environment for the project. The editable command should answer 1 as soon as the code changes, the one of `../wheel-test` 0 until the 0.2.0 wheel is installed.

```python
%%writefile src/thermometer/cli.py
import sys

from thermometer.conversion import celsius_to_fahrenheit


def main() -> int:
  arguments = sys.argv[1:]
  if not arguments:
    print(..., file=sys.stderr)
    return ...
  for argument in arguments:
    celsius = float(argument)
    print(f"{celsius:.1f} °C = {celsius_to_fahrenheit(celsius):.1f} °F")
  return ...
```

```python
!to-fahrenheit; echo "Exit code: $?"
!uv version ...
!uv build
!uv pip install --python ../wheel-test/bin/python dist/...
```

### Solution

In [ ]:
%%writefile src/thermometer/cli.py
import sys

from thermometer.conversion import celsius_to_fahrenheit


def main() -> int:
  arguments = sys.argv[1:]
  if not arguments:
    print("Usage: to-fahrenheit TEMPERATURE...", file=sys.stderr)
    return 1
  for argument in arguments:
    celsius = float(argument)
    print(f"{celsius:.1f} °C = {celsius_to_fahrenheit(celsius):.1f} °F")
  return 0

In [ ]:
!to-fahrenheit; echo "Exit code: $?"
!../wheel-test/bin/to-fahrenheit; echo "Exit code: $?"

In [ ]:
!uv version --bump minor --no-sync
!uv build
!uv pip install --python ../wheel-test/bin/python dist/thermometer-0.2.0-py3-none-any.whl
!../wheel-test/bin/to-fahrenheit; echo "Exit code: $?"

The editable install follows the sources; the `../wheel-test` environment keeps the copy of the 0.1.0 wheel until the 0.2.0 one is installed into it, which `uv pip install` puts in place of the old one. A new published version always has a new number: PyPI refuses to replace the files of a version. `dist/` keeps the files of both versions, and `uv publish` uploads everything it holds: empty it before building a version to publish.

## 7. Going Further: Publish on Test PyPI

This step is optional: it needs an account on [Test PyPI](https://test.pypi.org/account/register/), Python's test index, separate from PyPI, with a verified e-mail address and two-factor authentication, without which Test PyPI refuses any upload. Create that token in your account's settings.

The `thermometer` name is probably taken on Test PyPI already: in `pyproject.toml`, which you edit in the file editor (Colab's Files panel, the Codespace's explorer), rename the project `thermometer-<your username>`, without renaming the `src/thermometer` folder, which you then point `uv_build` to. Also declare Test PyPI there as a publishing index, then rebuild the package in an empty `dist/` and publish it with `uv publish`. In the notebook, don't type the token in clear: ask for it with `getpass`.

**Useful documentation**

- https://docs.astral.sh/uv/concepts/build-backend/#modules
- https://docs.astral.sh/uv/guides/package/#publishing-your-package
- https://docs.astral.sh/uv/concepts/indexes/#pinning-a-package-to-an-index
- https://docs.astral.sh/uv/reference/cli/#uv-publish--index
- https://docs.python.org/3/library/getpass.html#getpass.getpass

In [ ]:
# Your code here

### Additional hints

The `[tool.uv.build-backend]` table has a `module-name` key, the name of the module to put in the wheel: otherwise `uv_build` looks for `src/thermometer_<your_username>/`. An index is declared in a `[[tool.uv.index]]` table: a name (`testpypi`), the read `url` (`https://test.pypi.org/simple/`), the `publish-url` (`https://test.pypi.org/legacy/`) and `explicit = true`, so that uv never looks for the project's dependencies there. `uv publish --index testpypi --token {token}` publishes the content of `dist/` on that index (IPython replaces `{token}` with the variable's value). The solution's cell only publishes if you set `PUBLISH` to `True`.

```toml
[tool.uv.build-backend]
module-name = ...

[[tool.uv.index]]
name = "testpypi"
url = ...
publish-url = ...
explicit = true
```

```python
!rm -rf dist
!uv build
token = getpass(...)
!uv publish ...
```

### Solution

The `pyproject.toml`, with your username:

```toml
[project]
name = "thermometer-myusername"
version = "0.2.0"
description = "Temperature conversions."
readme = "README.md"
requires-python = ">=3.12"
dependencies = []

[project.scripts]
to-fahrenheit = "thermometer.cli:main"

[build-system]
requires = ["uv_build>=0.13.1,<0.14.0"]
build-backend = "uv_build"

[tool.uv.build-backend]
module-name = "thermometer"

[[tool.uv.index]]
name = "testpypi"
url = "https://test.pypi.org/simple/"
publish-url = "https://test.pypi.org/legacy/"
explicit = true
```

In [ ]:
from getpass import getpass

PUBLISH = False  # Set to True to publish on Test PyPI

if PUBLISH:
  !rm -rf dist
  !uv build
  token = getpass("Test PyPI API token: ")
  !uv publish --index testpypi --token {token}

The package then installs from another environment with `uv pip install --index https://test.pypi.org/simple/ thermometer-<your username>`. For PyPI itself, publishing is usually done from the CI, without a token, through Trusted Publishing: https://docs.pypi.org/trusted-publishers/.